In [ ]:
### define setting to indicate raw data in saved files
database <- "MGnify" # MGnify / IGC / Qin
grouping <- "Groups" # Groups / Subgroups
new_discovery_study <- "Werner" # ValdesMas / Werner
file_prefix <- paste(database,"_",grouping,"_",new_discovery_study,"_", sep ="")
print(file_prefix)

In [ ]:
if (!requireNamespace("openxlsx", quietly = TRUE))
  install.packages("openxlsx")
library(openxlsx)
file_name <- paste(file_prefix,"shared_metaproteins(discovery_studies_normalized_5).csv",sep ="")
Raw_Data <- read.delim(file_name,sep= ",")
names(Raw_Data)[1] <- "Metaprotein.Number"
rownames(Raw_Data) <- Raw_Data$Metaprotein.Number 
Raw_Data$Metaprotein.Number <- NULL


### Adjusting of column names for matching of measurement IDs and metadata IDs
#names(Raw_Data) <- sub(".*___(.*)\\_mgf.*", "\\1", names(Raw_Data))
names(Raw_Data) <- sub("^quant_[^_]+_(.*?)_mgf_.*$", "\\1", names(Raw_Data))
names(Raw_Data) <- sub("^quant_F[^_]+_(.*)_\\s*$", "\\1", names(Raw_Data))
### for qin dataset:
#names(Raw_Data) <- sub("^[^_]+_(.*)_[^_]+$", "\\1", names(Raw_Data))

names(Raw_Data) <- gsub("\\.", "\\_", names(Raw_Data))
names(Raw_Data) <- gsub("\\-", "\\_", names(Raw_Data))
names(Raw_Data) <- gsub("\\_480Ex2", "\\1", names(Raw_Data))

### load metadata
metadata_file <- read.xlsx("metadata_table_new_revision.xlsx")

metadata_file$ID <- gsub("\\.mgf.dat$", "", metadata_file$ID)
metadata_file$ID <- gsub("\\.", "\\_", metadata_file$ID)
metadata_file$ID <- gsub("\\-", "\\_", metadata_file$ID)

length(unique(rownames(Raw_Data)))

sum(is.na(Raw_Data))

sum(Raw_Data==0)

### MMUPHin Batch Correction

In [ ]:
### MMUPHin Batch Effect Removal
#BiocManager::install("MMUPHin")

library(MMUPHin)
library(MMUPHin)
library(magrittr)
library(dplyr)
library(ggplot2)


matrix_df <- Raw_Data
studies_discovery <- c("Henry", "Thuy-Boun", "Lehmann", "Lloyd-Price", "Werner")
metadata_file_Discovery <- metadata_file[metadata_file$study %in% studies_discovery,]
remove_diseases2 <- c("GCA", "IBS", "CA")
metadata_file_Discovery <- metadata_file_Discovery[!metadata_file_Discovery$disease %in% remove_diseases2,]
rownames(metadata_file_Discovery) <- metadata_file_Discovery$ID ## Samples sollen Zeilennamen sein
metadata_file_Discovery$ID <- NULL

length(colnames(Raw_Data))
length(rownames(metadata_file_Discovery))

### Check setdiff to identify measurements and metadata not matched
setdiff(colnames(Raw_Data), rownames(metadata_file_Discovery))
setdiff(rownames(metadata_file_Discovery), colnames(Raw_Data))

matrix_df <- matrix_df[, rownames(metadata_file_Discovery)]

all(colnames(matrix_df) == rownames(metadata_file_Discovery)) ## prüfen, ob Reihenfolge der Samples übereinstimmt

MMUPHin_data <- adjust_batch(feature_abd = matrix_df, batch = "study",covariates = "condition", data = metadata_file_Discovery)

In [ ]:
# save to .csv
print(head(MMUPHin_data$feature_abd_adj))
adjusted_df <- MMUPHin_data$feature_abd_adj
adjusted_df <- adjusted_df[, colnames(Raw_Data)]
file_name <- paste(file_prefix,"shared_metaproteins(discovery_studies_normalized_MMUPHin_corrected).csv",sep ="")
write.csv(adjusted_df, file_name, row.names = TRUE)

### ComBat Batch Effect Correction

In [ ]:
# ### ComBat Batch Effect Removal
#if (!requireNamespace("BiocManager", quietly = TRUE))
#    install.packages("BiocManager")

if (!requireNamespace("sva", quietly = TRUE))
    BiocManager::install("sva")

library(sva)

metadata <- metadata_file_Discovery
abundance <- Raw_Data

common_samples <- intersect(rownames(metadata), colnames(abundance))

metadata_sub  <- metadata[common_samples, , drop = FALSE]
abundance_sub <- abundance[, common_samples, drop = FALSE]

# covariate to adjust for (--> study)
batch <- factor(metadata_sub$study)

# ComBat 
abundance_combat <- ComBat( dat = as.matrix(abundance_sub),batch = batch,
    mod = NULL,par.prior = TRUE,prior.plots = FALSE)

abundance_combat <- as.data.frame(abundance_combat)

rownames(abundance_combat) <- rownames(abundance_sub)
colnames(abundance_combat) <- colnames(abundance_sub)

# save to .csv
file_name <- paste(file_prefix,"shared_metaproteins(discovery_studies_normalized_ComBat_corrected).csv",sep ="")
write.csv(abundance_combat, file_name, row.names = TRUE, quote = FALSE)

In [ ]:
# ### limma Batch Effect Removal
#if (!requireNamespace("BiocManager", quietly = TRUE))
#    install.packages("BiocManager")
#if (!requireNamespace("limma", quietly = TRUE))
#    BiocManager::install("limma")

library(limma)

metadata <- metadata_file_Discovery
abundance <- Raw_Data

common_samples <- intersect(rownames(metadata), colnames(abundance))

metadata_sub  <- metadata[common_samples, , drop = FALSE]
abundance_sub <- abundance[, common_samples, drop = FALSE]

# covariate to adjust for (--> study)
batch <- factor(metadata_sub$study)

design <- model.matrix(~ 1, data = metadata_sub)

# limma
abundance_limma <- removeBatchEffect(
    x = as.matrix(abundance_sub),
    batch = batch,
    design = design
)
abundance_limma <- as.data.frame(abundance_limma)
rownames(abundance_limma) <- rownames(abundance_sub)
colnames(abundance_limma) <- colnames(abundance_sub)

# save to .csv
file_name <- paste(file_prefix,
    "shared_metaproteins(discovery_studies_normalized_limma_corrected).csv",
    sep = "")
write.csv(abundance_limma,file = file_name,row.names = TRUE,quote = FALSE)